In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

print("FINAL BUSINESS RECOMMENDATIONS")
print("=" * 60)

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "feature_engineered_dataset_final.csv"
RISK_PATH = PROJECT_ROOT / "reports" / "risk_scoring.csv"

print("Project Root:", PROJECT_ROOT.resolve())
print("Final Dataset:", DATA_PATH.exists())
print("Risk Scoring:", RISK_PATH.exists())

final_df = pd.read_csv(DATA_PATH)
risk_df = pd.read_csv(RISK_PATH)

print("\nFinal dataset rows:", len(final_df))
print("Risk scoring rows:", len(risk_df))

print("\nFiles loaded successfully.")

FINAL BUSINESS RECOMMENDATIONS
Project Root: C:\Users\user\Documents\Zidio_Project\FORESIGHT
Final Dataset: True
Risk Scoring: True

Final dataset rows: 73000
Risk scoring rows: 73000

Files loaded successfully.


C:\Users\user\AppData\Local\Temp\ipykernel_15880\3043725630.py:18: DtypeWarning: Columns (0: promo_event) have mixed types. Specify dtype option on import or set low_memory=False.
  risk_df = pd.read_csv(RISK_PATH)


In [2]:
# Step 2: Overall Business Risk Summary

print("OVERALL BUSINESS RISK SUMMARY")
print("=" * 60)

risk_summary = (
    risk_df["Risk_Level"]
    .value_counts()
    .reindex(["High", "Medium", "Low"], fill_value=0)
)

total_risk_rows = risk_summary.sum()

print("\nRisk distribution:")
print(risk_summary)

print("\nRisk percentages:")
for level, count in risk_summary.items():
    percentage = (count / total_risk_rows) * 100
    print(f"{level}: {count:,} rows ({percentage:.2f}%)")

high_risk_percentage = (
    risk_summary["High"] / total_risk_rows
) * 100

print("\nBusiness interpretation:")

if high_risk_percentage >= 50:
    print(
        f"High-risk inventory represents {high_risk_percentage:.2f}% "
        "of evaluated records. Immediate inventory monitoring and "
        "replenishment planning are recommended."
    )
else:
    print(
        f"High-risk inventory represents {high_risk_percentage:.2f}% "
        "of evaluated records. Focused monitoring of high-risk items "
        "is recommended."
    )

print("\nStep 2 completed.")

OVERALL BUSINESS RISK SUMMARY

Risk distribution:
Risk_Level
High      63400
Medium       51
Low        9549
Name: count, dtype: int64

Risk percentages:
High: 63,400 rows (86.85%)
Medium: 51 rows (0.07%)
Low: 9,549 rows (13.08%)

Business interpretation:
High-risk inventory represents 86.85% of evaluated records. Immediate inventory monitoring and replenishment planning are recommended.

Step 2 completed.


In [3]:
# Step 3: High-Risk SKU Analysis

print("HIGH-RISK SKU ANALYSIS")
print("=" * 60)

high_risk_df = risk_df[risk_df["Risk_Level"] == "High"].copy()

print("High-risk records:", len(high_risk_df))

sku_risk_summary = (
    high_risk_df
    .groupby("sku_id")
    .agg(
        High_Risk_Records=("Risk_Level", "size"),
        Average_Risk_Score=("Risk_Score", "mean"),
        Average_Predicted_Demand=("Predicted_Units_Sold", "mean"),
        Average_On_Hand=("on_hand_units", "mean"),
        Average_Inventory_Gap=("Inventory_Gap", "mean"),
        Average_Inventory_Coverage=("Inventory_Coverage", "mean")
    )
    .sort_values(
        ["High_Risk_Records", "Average_Risk_Score"],
        ascending=False
    )
)

print("\nTop 20 high-risk SKUs:")
print(sku_risk_summary.head(20).round(2))

print("\nHigh-risk SKU count:", len(sku_risk_summary))

print("\nBusiness recommendation:")
print(
    "Prioritize SKUs with frequent high-risk records, high risk scores, "
    "large inventory gaps, and low inventory coverage for immediate "
    "inventory review and replenishment planning."
)

print("\nStep 3 completed.")

HIGH-RISK SKU ANALYSIS
High-risk records: 63400

Top 20 high-risk SKUs:
        High_Risk_Records  Average_Risk_Score  Average_Predicted_Demand  \
sku_id                                                                    
SKU001                317               100.0                     12.12   
SKU002                317               100.0                     11.90   
SKU003                317               100.0                     11.73   
SKU004                317               100.0                     12.32   
SKU005                317               100.0                     11.94   
SKU006                317               100.0                     11.62   
SKU007                317               100.0                     12.09   
SKU008                317               100.0                     12.21   
SKU009                317               100.0                     12.18   
SKU010                317               100.0                     11.76   
SKU011                317   

In [4]:
# Step 4: Category-Level Business Recommendations

print("CATEGORY-LEVEL BUSINESS ANALYSIS")
print("=" * 60)

category_summary = (
    risk_df
    .groupby("category")
    .agg(
        Total_Records=("sku_id", "size"),
        High_Risk_Records=("Risk_Level", lambda x: (x == "High").sum()),
        Medium_Risk_Records=("Risk_Level", lambda x: (x == "Medium").sum()),
        Low_Risk_Records=("Risk_Level", lambda x: (x == "Low").sum()),
        Average_Risk_Score=("Risk_Score", "mean"),
        Average_Predicted_Demand=("Predicted_Units_Sold", "mean"),
        Average_On_Hand=("on_hand_units", "mean"),
        Average_Inventory_Gap=("Inventory_Gap", "mean")
    )
)

category_summary["High_Risk_Percentage"] = (
    category_summary["High_Risk_Records"]
    / category_summary["Total_Records"]
    * 100
)

category_summary = category_summary.sort_values(
    "High_Risk_Percentage",
    ascending=False
)

print("\nCategory risk summary:")
print(category_summary.round(2).to_string())

print("\nBusiness recommendations by category:")

for category, row in category_summary.iterrows():

    high_pct = row["High_Risk_Percentage"]

    if high_pct >= 80:
        recommendation = (
            "Priority action: closely monitor inventory, review reorder "
            "levels, and ensure timely replenishment."
        )
    elif high_pct >= 50:
        recommendation = (
            "Moderate priority: review demand and inventory levels "
            "regularly and adjust replenishment where required."
        )
    else:
        recommendation = (
            "Lower priority: maintain monitoring and optimize inventory "
            "based on demand trends."
        )

    print(f"\n{category}: {recommendation}")

print("\nStep 4 completed.")

CATEGORY-LEVEL BUSINESS ANALYSIS

Category risk summary:
            Total_Records  High_Risk_Records  Medium_Risk_Records  Low_Risk_Records  Average_Risk_Score  Average_Predicted_Demand  Average_On_Hand  Average_Inventory_Gap  High_Risk_Percentage
category                                                                                                                                                                                       
Appliances          17885              15533                   19              2333               88.21                     11.91            21.09                   9.18                 86.85
Decor               18980              16484                    9              2487               88.23                     11.90            20.70                   8.80                 86.85
Furniture           20075              17435                   16              2624               88.24                     11.86            20.76                   8.90      

In [6]:
# Step 5: Inventory and Replenishment Recommendations

print("INVENTORY & REPLENISHMENT RECOMMENDATIONS")
print("=" * 60)

# Identify records requiring immediate attention
high_priority = risk_df[
    (risk_df["Risk_Level"] == "High") &
    (
        (risk_df["Inventory_Gap"] > 0) |
        (risk_df["Inventory_Coverage"] < 1)
    )
].copy()

print("High-priority inventory records:", len(high_priority))

print("\nAverage values for high-priority records:")
print(
    high_priority[
        [
            "Predicted_Units_Sold",
            "on_hand_units",
            "Inventory_Gap",
            "Inventory_Coverage",
            "Risk_Score"
        ]
    ].mean().round(2)
)

# Top SKUs requiring attention based on average inventory gap
replenishment_summary = (
    high_priority
    .groupby("sku_id")
    .agg(
        Priority_Records=("sku_id", "size"),
        Average_Predicted_Demand=("Predicted_Units_Sold", "mean"),
        Average_On_Hand=("on_hand_units", "mean"),
        Average_Inventory_Gap=("Inventory_Gap", "mean"),
        Average_Coverage=("Inventory_Coverage", "mean"),
        Average_Risk_Score=("Risk_Score", "mean")
    )
    .sort_values(
        ["Average_Inventory_Gap", "Average_Risk_Score"],
        ascending=False
    )
)

print("\nTop 20 SKUs requiring replenishment attention:")
print(replenishment_summary.head(20).round(2).to_string())

print("\nFINAL REPLENISHMENT RECOMMENDATION")
print(
    "High-risk SKUs show zero on-hand inventory, zero inventory coverage, "
    "and a negative inventory gap relative to predicted demand. These SKUs "
    "should receive immediate replenishment attention. Reorder points and "
    "on-order quantities should be reviewed to reduce potential stockout risk."
)

print("\nStep 5 completed.")

INVENTORY & REPLENISHMENT RECOMMENDATIONS
High-priority inventory records: 63400

Average values for high-priority records:
Predicted_Units_Sold     11.9
on_hand_units             0.0
Inventory_Gap           -11.9
Inventory_Coverage        0.0
Risk_Score              100.0
dtype: float64

Top 20 SKUs requiring replenishment attention:
        Priority_Records  Average_Predicted_Demand  Average_On_Hand  Average_Inventory_Gap  Average_Coverage  Average_Risk_Score
sku_id                                                                                                                          
SKU192               317                     11.22              0.0                 -11.22               0.0               100.0
SKU103               317                     11.38              0.0                 -11.38               0.0               100.0
SKU054               317                     11.39              0.0                 -11.39               0.0               100.0
SKU194            

In [7]:
# Step 6: Final Executive Business Recommendations

print("FINAL EXECUTIVE BUSINESS RECOMMENDATIONS")
print("=" * 60)

print("\n1. INVENTORY REPLENISHMENT")
print(
    "Immediately prioritize high-risk SKUs with zero on-hand inventory "
    "and zero inventory coverage. Review reorder points and pending "
    "purchase orders to reduce stockout exposure."
)

print("\n2. HIGH-RISK SKU MONITORING")
print(
    "Use the risk score to continuously monitor high-risk SKUs. "
    "SKUs with persistent Risk_Score values near 100 should receive "
    "the highest operational priority."
)

print("\n3. DEMAND-BASED PLANNING")
print(
    "Use predicted units sold from the final Random Forest model "
    "to support inventory planning, replenishment decisions, "
    "and future demand estimation."
)

print("\n4. CATEGORY MANAGEMENT")
print(
    "All four categories show substantial high-risk exposure. "
    "Appliances, Decor, Furniture, and Kitchen should therefore "
    "be monitored using category-level inventory and demand reports."
)

print("\n5. INVENTORY COVERAGE")
print(
    "Maintain sufficient inventory coverage against predicted demand. "
    "Items with zero coverage should be treated as immediate "
    "replenishment candidates."
)

print("\n6. OPERATIONAL MONITORING")
print(
    "Integrate demand forecasts and risk scores into the dashboard "
    "so inventory teams can identify priority SKUs quickly and "
    "take corrective action."
)

print("\n7. MANAGEMENT ACTION")
print(
    "Business teams should review high-risk inventory regularly, "
    "adjust replenishment plans according to predicted demand, "
    "and monitor changes in inventory risk over time."
)

print("\n" + "=" * 60)
print("FINAL BUSINESS RECOMMENDATIONS COMPLETED")
print("=" * 60)

FINAL EXECUTIVE BUSINESS RECOMMENDATIONS

1. INVENTORY REPLENISHMENT
Immediately prioritize high-risk SKUs with zero on-hand inventory and zero inventory coverage. Review reorder points and pending purchase orders to reduce stockout exposure.

2. HIGH-RISK SKU MONITORING
Use the risk score to continuously monitor high-risk SKUs. SKUs with persistent Risk_Score values near 100 should receive the highest operational priority.

3. DEMAND-BASED PLANNING
Use predicted units sold from the final Random Forest model to support inventory planning, replenishment decisions, and future demand estimation.

4. CATEGORY MANAGEMENT
All four categories show substantial high-risk exposure. Appliances, Decor, Furniture, and Kitchen should therefore be monitored using category-level inventory and demand reports.

5. INVENTORY COVERAGE
Maintain sufficient inventory coverage against predicted demand. Items with zero coverage should be treated as immediate replenishment candidates.

6. OPERATIONAL MONITORING